<!-- NOTEBOOK_METADATA source: "Jupyter Notebook" title: "Optimize LLM Applications with Weco AI" sidebarTitle: "Weco AI" description: "Automatically optimize LLM application code using Weco's AI-driven optimization with Langfuse experiments for evaluation and comparison." category: "Integrations" -->

# Optimize LLM Applications with Weco + Langfuse

[Weco AI](https://www.weco.ai/) is an AI-powered code optimizer that iteratively improves your code against measurable metrics. It can optimize prompts, agent logic, tool configurations, and any other code artifact.

[Langfuse](https://langfuse.com/) is an open-source LLM engineering platform for tracing, evaluation, and experimentation.

This notebook shows how to use **Weco with Langfuse as the evaluation backend** to automatically optimize an LLM application. Langfuse provides:
- **Datasets** to hold test cases
- **Experiments** to run and compare application variants
- **Managed evaluators** (LLM-as-a-Judge) for server-side quality assessment
- **Local evaluators** for deterministic checks

Weco iteratively generates improved code variants and uses Langfuse experiments to evaluate each one, converging on the best-performing version.

### How It Works

1. You define a **target function** (your LLM application) and **evaluators** (quality metrics)
2. Weco's optimizer generates improved code variants (e.g., better prompts)
3. Each variant is evaluated against your Langfuse dataset using both local and managed evaluators
4. Weco keeps the best variant and iterates until the metric converges
5. All experiments are tracked in Langfuse for comparison and analysis

## Step 1: Install Dependencies

In [ ]:
%pip install 'weco[langfuse]' langfuse openai --upgrade --quiet

## Step 2: Set Up Environment Variables

Set up your **Langfuse** API keys ([Langfuse Cloud](https://cloud.langfuse.com) or [self-hosted](https://langfuse.com/self-hosting)), your **OpenAI** API key, and your **Weco** API key ([weco.ai](https://www.weco.ai/)).

In [ ]:
import os

# Get keys for your project from the project settings page: https://cloud.langfuse.com
os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
os.environ["LANGFUSE_BASE_URL"] = "https://cloud.langfuse.com"  # 🇪🇺 EU region
# os.environ["LANGFUSE_BASE_URL"] = "https://us.cloud.langfuse.com"  # 🇺🇸 US region

# LLM provider
os.environ["OPENAI_API_KEY"] = "sk-..."

# Weco API key (https://www.weco.ai/)
os.environ["WECO_API_KEY"] = "..."

In [ ]:
from langfuse import get_client

langfuse = get_client()

# Verify connection
if langfuse.auth_check():
    print("Langfuse client is authenticated and ready!")
else:
    print("Authentication failed. Please check your credentials and host.")

## Step 3: Create a Dataset

We'll create a Langfuse dataset with HR policy questions from a fictional company called ZephHR. Each item has an input question and an expected answer for evaluation.

In [ ]:
DATASET_NAME = "zephhr-qa-opt"

# Create the dataset (idempotent)
try:
    dataset = langfuse.create_dataset(
        name=DATASET_NAME,
        description="ZephHR QA optimization set"
    )
    print(f"Created dataset: {DATASET_NAME}")
except Exception:
    dataset = langfuse.get_dataset(DATASET_NAME)
    print(f"Using existing dataset: {DATASET_NAME}")

# Sample questions
questions = [
    {
        "id": "opt-001",
        "question": "An employee was rehired 10 months after leaving. Do they need to redo full onboarding?",
        "expected_answer": "No. Employees rehired within 12 months of their termination date may use a streamlined re-hire flow that preserves their previous tax elections."
    },
    {
        "id": "opt-002",
        "question": "Our company is on the Starter plan. Can we set up SSO?",
        "expected_answer": "SSO/SAML is not available on the Starter plan. It is available on the Professional plan as an add-on at $2 per user per month, and it is included with the Enterprise plan."
    },
    {
        "id": "opt-003",
        "question": "I'm a manager. Can I see how much my direct report makes?",
        "expected_answer": "No. Managers cannot view or edit compensation details for their reports. Only HR admins have access to compensation information."
    },
    {
        "id": "opt-004",
        "question": "We need to issue a bonus to an employee. What's the process?",
        "expected_answer": "Bonuses are processed as off-cycle payments. An HR admin must request the off-cycle payment, and it requires VP-level approval."
    },
    {
        "id": "opt-005",
        "question": "When does a new full-time employee become eligible for medical benefits?",
        "expected_answer": "Full-time employees (30+ hours/week) become eligible for benefits on the first day of the month following 60 days of employment."
    }
]

# Add items to dataset
for q in questions:
    langfuse.create_dataset_item(
        dataset_name=DATASET_NAME,
        input={"question": q["question"]},
        expected_output={"expected_answer": q["expected_answer"]},
        metadata={"case_id": q["id"]},
    )

langfuse.flush()
print(f"Added {len(questions)} items to dataset '{DATASET_NAME}'")

## Step 4: Define the Target Function

The target function is the LLM application that Weco will optimize. It takes a dict of inputs and returns a dict of outputs.

In this example, it's a QA agent that answers HR policy questions using a knowledge base. Weco will optimize the `SYSTEM_PROMPT` and `USER_TEMPLATE`.

> **Note:** We use `%%writefile` to write the code to disk because Weco optimizes source files directly — it reads, modifies, and rewrites them during each optimization step.

In [ ]:
%%writefile agent.py
"""Baseline QA agent for ZephHR documentation.

Answers HR policy questions using only the docs.md knowledge base.
Weco optimizes this file — specifically the SYSTEM_PROMPT and USER_TEMPLATE.
"""

import json
from pathlib import Path

from openai import OpenAI

client = OpenAI()

DOCS = Path(__file__).with_name("docs.md").read_text()

SYSTEM_PROMPT = """You are a ZephHR support assistant. Answer the user's question
using ONLY the provided documentation. Do not guess or invent policy details.

If the documentation does not contain enough information to fully answer,
say so clearly and state what IS covered.

Return your answer as JSON with exactly these fields:
- answer: your response to the question (string)
- confidence: how confident you are the answer is fully supported by the docs (high/medium/low)
- relevant_sections: list of section names from the docs you referenced"""

USER_TEMPLATE = """Documentation:
{docs}

Question: {question}

Return only JSON."""


def answer_hr_question(inputs: dict) -> dict:
    """Answer an HR policy question from the ZephHR docs."""
    question = inputs.get("question", "")

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_TEMPLATE.format(docs=DOCS, question=question)},
        ],
        temperature=0.0,
        response_format={"type": "json_object"},
    )

    try:
        parsed = json.loads(response.choices[0].message.content)
    except (TypeError, json.JSONDecodeError):
        parsed = {}

    confidence = parsed.get("confidence", "low")
    if confidence not in ("high", "medium", "low"):
        confidence = "low"

    relevant_sections = parsed.get("relevant_sections", [])
    if not isinstance(relevant_sections, list):
        relevant_sections = []

    return {"answer": parsed.get("answer", ""), "confidence": confidence, "relevant_sections": relevant_sections}

## Step 5: Define Evaluators

Evaluators score each output from the target function. Langfuse evaluators receive keyword arguments (`input`, `output`, `expected_output`, `metadata`) and return an `Evaluation` object.

We define two **local evaluators** (deterministic checks) and a **metric function** that combines all scores:

In [ ]:
%%writefile evaluators.py
"""Evaluators and metric function for ZephHR QA (Langfuse format).

Code evaluators (run locally):
- json_schema_validity: checks the agent output has the required JSON schema
- conciseness: penalises excessively long or empty answers

LLM judges (configured in Langfuse UI as managed evaluators):
- Helpfulness: how helpful and complete the answer is (0-1 scale)
- Correctness: binary factual accuracy (0 or 1)

Metric function:
- qa_score: multiplies correctness (gate) by helpfulness (signal)
"""

from langfuse import Evaluation


def json_schema_validity(*, input, output, expected_output=None, **kwargs):
    """Check that the agent output contains the required fields with correct types."""
    outputs = output or {}

    checks = {
        "answer": isinstance(outputs.get("answer"), str) and len(outputs["answer"]) > 0,
        "confidence": outputs.get("confidence") in ("high", "medium", "low"),
        "relevant_sections": isinstance(outputs.get("relevant_sections"), list),
    }

    passed = all(checks.values())
    failed_fields = [k for k, v in checks.items() if not v]
    comment = "All fields valid" if passed else f"Invalid fields: {', '.join(failed_fields)}"

    return Evaluation(name="json_schema_validity", value=1.0 if passed else 0.0, comment=comment)


def conciseness(*, input, output, expected_output=None, **kwargs):
    """Score based on answer length \u2014 penalise empty or excessively verbose answers."""
    answer = (output or {}).get("answer", "")
    word_count = len(answer.split())

    if word_count == 0:
        score, comment = 0.0, "Empty answer"
    elif word_count <= 150:
        score, comment = 1.0, f"{word_count} words \u2014 concise"
    elif word_count <= 250:
        score, comment = 0.5, f"{word_count} words \u2014 verbose"
    else:
        score, comment = 0.0, f"{word_count} words \u2014 excessively long"

    return Evaluation(name="conciseness", value=score, comment=comment)


def qa_score(scores: dict) -> float:
    """Combine correctness (binary gate) with helpfulness (0-1 signal).

    correctness * helpfulness

    - Incorrect answers always score 0.
    - Correct answers are ranked by helpfulness.
    """
    correctness = scores.get("Correctness", 0.0)
    helpfulness = scores.get("Helpfulness", 0.0)
    return correctness * helpfulness

## Step 6: Configure Managed Evaluators

In addition to local evaluators, Langfuse can run **managed evaluators** (LLM-as-a-Judge) server-side. These are configured in the Langfuse UI and automatically score experiment traces.

For this example, configure two managed evaluators in your Langfuse project:

1. Go to your Langfuse project → **Evaluation** → **Evaluators**
2. Click **+ New Evaluator** for each evaluator below

### Correctness evaluator

- **Name**: `Correctness`
- **Score**: 0 or 1 (binary factual accuracy)
- **Variable mappings** (JSONPath expressions pointing to trace fields):
  - `{{input}}` → `$.input.question` (the user's question)
  - `{{output}}` → `$.output.answer` (the agent's answer)
  - `{{expected_output}}` → `$.expected_output.expected_answer` (the ground truth)

### Helpfulness evaluator

- **Name**: `Helpfulness`
- **Score**: 0–1 continuous scale
- **Variable mappings**:
  - `{{input}}` → `$.input.question`
  - `{{output}}` → `$.output.answer`

Use the **live preview** to verify the mappings are picking up the correct data from your traces.

> **Important:** Evaluator names are case-sensitive. The names you set in the Langfuse UI (e.g., `Correctness`, `Helpfulness`) must match exactly what you pass to `--langfuse-managed-evaluators` and use as keys in your metric function (`scores.get("Correctness")`).

Weco's bridge will automatically poll for these server-side scores after each experiment run.

## Step 7: Run a Baseline Experiment

Before optimizing, let's run a baseline experiment to see how the unoptimized agent performs. This uses Langfuse's experiment runner SDK directly.

In [ ]:
from agent import answer_hr_question
from evaluators import json_schema_validity, conciseness

dataset = langfuse.get_dataset(DATASET_NAME)


def task(*, item, **kwargs):
    """Wrap the target function for Langfuse's experiment runner."""
    return answer_hr_question(item.input)


baseline_result = dataset.run_experiment(
    name="zephhr-baseline",
    task=task,
    evaluators=[json_schema_validity, conciseness],
)

print(baseline_result.format())

You can view the full experiment results in the [Langfuse dashboard](https://cloud.langfuse.com) under your project's **Datasets** → **zephhr-qa-opt** → **Runs**.

## Step 8: Optimize with Weco

Now let's use Weco to automatically optimize the agent. Weco will:

1. Read the source code (`agent.py`)
2. Generate improved variants (modifying prompts, logic, etc.)
3. Evaluate each variant using Langfuse experiments
4. Keep the best-performing variant and iterate

The `--eval-backend langfuse` flag tells Weco to use the Langfuse bridge. Local evaluators (`json_schema_validity`, `conciseness`) run in-process, while managed evaluators (`Correctness`, `Helpfulness`) are polled from Langfuse's server-side LLM-as-a-Judge.

In [ ]:
!weco run --source agent.py \
  --eval-backend langfuse \
  --langfuse-dataset zephhr-qa-opt \
  --langfuse-target agent:answer_hr_question \
  --langfuse-evaluators evaluators:json_schema_validity evaluators:conciseness \
  --langfuse-managed-evaluators Correctness Helpfulness \
  --langfuse-metric-function evaluators:qa_score \
  --metric qa_score --goal maximize --steps 10

## Step 9: Compare Results in Langfuse

After optimization, Weco has written the best variant back to `agent.py`. Each optimization step created a Langfuse experiment, so you can compare all variants in the Langfuse dashboard.

Let's run the optimized agent as a final experiment for side-by-side comparison:

In [ ]:
import importlib
import agent

# Reload to pick up the optimized version
importlib.reload(agent)

dataset = langfuse.get_dataset(DATASET_NAME)


def optimized_task(*, item, **kwargs):
    return agent.answer_hr_question(item.input)


optimized_result = dataset.run_experiment(
    name="zephhr-weco-optimized",
    task=optimized_task,
    evaluators=[json_schema_validity, conciseness],
)

print(optimized_result.format())

Navigate to **Datasets** → **zephhr-qa-opt** in the [Langfuse dashboard](https://cloud.langfuse.com) to compare the baseline and optimized experiment runs side-by-side.

## Next Steps

- **Add more evaluators**: Create custom LLM-as-a-Judge evaluators in the Langfuse UI or write additional local evaluators
- **Use larger datasets**: Add more test cases to improve optimization reliability
- **Holdout validation**: Create a separate dataset (`zephhr-qa-holdout`) and run a single-step evaluation to verify the optimized agent generalizes
- **Explore the Weco dashboard**: View optimization history and compare variants at [dashboard.weco.ai](https://dashboard.weco.ai)

## Learn More

- [Weco Documentation](https://docs.weco.ai/)
- [Weco CLI on GitHub](https://github.com/WecoAI/weco-cli)
- [Langfuse Experiments Documentation](https://langfuse.com/docs/evaluation/experiments)
- [Langfuse Evaluator Library](https://langfuse.com/docs/evaluation/evaluation-methods/llm-as-a-judge)
- [Langfuse Datasets](https://langfuse.com/docs/datasets)

<!-- MARKDOWN_COMPONENT name: "LearnMore" path: "@/components-mdx/integration-learn-more.mdx" -->